# Experiment: Delay Time Distribution

Objective:
- Load the delay-time distribution stored in `delta_days_distribution.npz`.
- Split the latest HDF5 delays by event-level realized ToO status.
- Summarize the Baseline, ToO Follow-up, and combined delay-time arrays.
- Produce a publication-ready delay-time distribution figure and save it under `gw-kn-multimodal/figures/`.


In [1]:
import os
_BASE = os.environ.get('BASE_DIR', '/fred/oz016/bgao_kn')

from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE = Path(f"{_BASE}/gw-kn-multimodal")
NPZ_PATH = Path(f"{_BASE}/data/Optical_Only_dataset/delta_days_distribution.npz")
ALBEF_H5 = Path(f"{_BASE}/data/ALBEF_dataset/combined_dataset_train.h5")
TOO_LABELS_CSV = BASE / "analysis" / "too_delay_diagnostic_20260823" / "event_level_too_delay.csv"
OUTDIR = BASE / "figures" / "delay_time_distribution"
OUTDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": 19,
    "axes.labelsize": 21,
    "axes.titlesize": 21,
    "xtick.labelsize": 19,
    "ytick.labelsize": 19,
    "legend.fontsize": 19,
    "figure.dpi": 300,
})

NPZ_PATH, ALBEF_H5, TOO_LABELS_CSV, OUTDIR

(PosixPath('/fred/oz016/bgao_kn/data/Optical_Only_dataset/delta_days_distribution.npz'),
 PosixPath('/fred/oz016/bgao_kn/data/ALBEF_dataset/combined_dataset_train.h5'),
 PosixPath('/fred/oz016/bgao_kn/gw-kn-multimodal/analysis/too_delay_diagnostic_20260823/event_level_too_delay.csv'),
 PosixPath('/fred/oz016/bgao_kn/gw-kn-multimodal/figures/delay_time_distribution'))

## Plan

- Read the combined delay-time array and metadata from the NPZ file.
- Use the latest ALBEF HDF5 and event-level ToO labels to split Baseline and ToO Follow-up samples.
- Verify that the HDF5-derived combined delays match the canonical NPZ exactly.
- Compute a compact statistical summary.
- Plot normalized histograms for Baseline, ToO Follow-up, and the combined sample on the same axes.
- Mark `\Delta t = 0` to distinguish negative and positive delays.


In [2]:
with np.load(NPZ_PATH, allow_pickle=False) as data:
    combined = np.asarray(data["delta_days_combined"], dtype=np.float64)
    snr_threshold = float(np.asarray(data["snr_detection_threshold"]))
    merge_window_hours = float(np.asarray(data["merge_window_hours"]))

too_labels = pd.read_csv(TOO_LABELS_CSV, usecols=["gw_idx", "realized_too"])
if too_labels["gw_idx"].duplicated().any():
    raise ValueError("Duplicate gw_idx values found in ToO event labels")

with h5py.File(ALBEF_H5, "r") as handle:
    event_time_mjd = np.asarray(handle["events/gw_data/event_time_mjd"][:], dtype=np.float64)
    parent_gw_idx = np.asarray(handle["events/optical_data/parent_gw_idx"][:], dtype=np.int64)
    first_detection_mjd = np.asarray(handle["events/optical_data/first_detection_mjd"][:], dtype=np.float64)

event_is_too = np.zeros(event_time_mjd.size, dtype=bool)
event_is_labeled = np.zeros(event_time_mjd.size, dtype=bool)
label_indices = too_labels["gw_idx"].to_numpy(dtype=np.int64)
event_is_too[label_indices] = too_labels["realized_too"].to_numpy(dtype=bool)
event_is_labeled[label_indices] = True
if not np.all(event_is_labeled[parent_gw_idx]):
    raise ValueError("At least one optical realization has no event-level ToO label")

h5_combined = (first_detection_mjd - event_time_mjd[parent_gw_idx]).astype(np.float32)
if not np.array_equal(combined.astype(np.float32), h5_combined):
    raise ValueError("HDF5-derived delays do not match delta_days_combined in the NPZ")

too_mask = event_is_too[parent_gw_idx]
baseline = combined[~too_mask]
too = combined[too_mask]
if baseline.size + too.size != combined.size:
    raise RuntimeError("Baseline and ToO samples do not partition the combined array")

def summarize(population: str, values: np.ndarray) -> dict:
    return {
        "population": population,
        "count": values.size,
        "mean_days": np.mean(values),
        "median_days": np.median(values),
        "p05_days": np.quantile(values, 0.05),
        "p95_days": np.quantile(values, 0.95),
    }

summary = pd.DataFrame([
    summarize("Baseline", baseline),
    summarize("ToO Follow-up", too),
    summarize("Combined", combined),
])
summary

,population,count,mean_days,median_days,p05_days,p95_days
0,Baseline,939519,1.730828,1.431603,0.375884,4.092358
1,ToO Follow-up,8117942,0.608450,0.558765,0.137366,1.228272
2,Combined,9057461,0.724873,0.602391,0.142115,1.669556


In [3]:
def plot_delay_distribution(baseline: np.ndarray, too: np.ndarray, combined: np.ndarray, outpath: Path) -> Path:
    q_lo = float(np.quantile(combined, 0.001))
    q_hi = float(np.quantile(combined, 0.999))
    bins = np.linspace(q_lo, q_hi, 90)

    fig, ax = plt.subplots(figsize=(9.2, 5.8))
    ax.hist(
        combined,
        bins=bins,
        density=True,
        histtype="stepfilled",
        alpha=0.18,
        color="#374151",
        edgecolor="#374151",
        linewidth=1.0,
        label="Combined",
    )
    ax.hist(
        baseline,
        bins=bins,
        density=True,
        histtype="step",
        color="#2563EB",
        linewidth=2.0,
        label="Baseline",
    )
    ax.hist(
        too,
        bins=bins,
        density=True,
        histtype="step",
        color="#DC2626",
        linewidth=2.0,
        label="ToO Follow-up",
    )

    ax.axvline(0.0, color="#111827", linewidth=1.2, linestyle="--", alpha=0.8)
    ax.set_xlabel(r"Delay from GW trigger to first LSST detection (days)")
    ax.set_ylabel("Probability density")
    ax.legend(frameon=False, ncol=3, loc="upper right")

    fig.tight_layout()
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return outpath

## Generate Figure


In [4]:
png_path = OUTDIR / "delay_time_distribution.png"
pdf_path = OUTDIR / "delay_time_distribution.pdf"
svg_path = OUTDIR / "delay_time_distribution.svg"

plot_delay_distribution(baseline, too, combined, png_path)
plot_delay_distribution(baseline, too, combined, pdf_path)
plot_delay_distribution(baseline, too, combined, svg_path)

{
    "png": str(png_path),
    "pdf": str(pdf_path),
    "svg": str(svg_path),
}

{'png': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/delay_time_distribution/delay_time_distribution.png',
 'pdf': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/delay_time_distribution/delay_time_distribution.pdf',
 'svg': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/delay_time_distribution/delay_time_distribution.svg'}

In [5]:
result = {
    "npz_file": str(NPZ_PATH),
    "albef_h5": str(ALBEF_H5),
    "too_labels_csv": str(TOO_LABELS_CSV),
    "too_definition": "mode == baseline_plus_too and too_visits_written > 0",
    "snr_threshold": snr_threshold,
    "merge_window_hours": merge_window_hours,
    "png": str(png_path),
    "pdf": str(pdf_path),
    "svg": str(svg_path),
}
result

{'npz_file': '/fred/oz016/bgao_kn/data/Optical_Only_dataset/delta_days_distribution.npz',
 'albef_h5': '/fred/oz016/bgao_kn/data/ALBEF_dataset/combined_dataset_train.h5',
 'too_labels_csv': '/fred/oz016/bgao_kn/gw-kn-multimodal/analysis/too_delay_diagnostic_20260823/event_level_too_delay.csv',
 'too_definition': 'mode == baseline_plus_too and too_visits_written > 0',
 'snr_threshold': 5.0,
 'merge_window_hours': 2.0,
 'png': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/delay_time_distribution/delay_time_distribution.png',
 'pdf': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/delay_time_distribution/delay_time_distribution.pdf',
 'svg': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/delay_time_distribution/delay_time_distribution.svg'}

## Results

- The notebook verifies HDF5-derived delays against `delta_days_combined` in the canonical NPZ.
- The output figure overlays Baseline and ToO Follow-up distributions with the combined distribution as a shaded reference.
- PNG, PDF, and SVG versions are saved under `gw-kn-multimodal/figures/delay_time_distribution/`.
